# Yahtzee y Montecarlo

Adaptación del cuaderno del profesor. Recorrido **USA, MODIFICA y CREA**. La partida clásica usa dos jugadores automáticos y dados uniformes. Los dados cargados se estudian por separado.

Ejecute en orden desde la carpeta del proyecto; conserve junto al cuaderno los módulos Python. Los materiales originales están en referencias/.

In [2]:
import json
from pathlib import Path
from pprint import pprint
from yahtzee import play_game, print_scoreboard, score_hand, FAIR, LOADED
from experimentos import exact_fixed_ev, theoretical_yahtzee, exact_risk, run_experiments

SEED = 42
SIMS_PER_MASK = 100
N = 100000
print('Semilla:', SEED, '| Simulaciones por máscara:', SIMS_PER_MASK, '| Muestras por experimento:', N)

Semilla: 42 | Simulaciones por máscara: 100 | Muestras por experimento: 100000


## 1. USA: ejecutar y comprender el código base

Se ejecutó la celda 0 original. Se conserva su salida completa en resultados/ejecucion_base.txt. No se ejecutan de nuevo sus celdas duplicadas al importar el motor adaptado.

La idea central se conserva: generar dados, evaluar máscaras de bloqueo con Montecarlo, relanzar los dados libres y anotar en una casilla vacía.

![Flujo original](resultados/flujo_base.svg)

In [3]:
diagnostico = json.loads(Path('resultados/diagnostico_base.json').read_text(encoding='utf-8'))
pprint(diagnostico)
print('El presupuesto efectivo original seguía en 300 aunque se cambiara la variable global.')

{'cell_executed': 0,
 'effective_default_sims': 300,
 'global_sims_after_change': 10,
 'last_reroll_mask': [False, False, False, False, False],
 'original_sha256': '87fc83ef866efa3ca649c38a7c29de50b239e4e4ba59c63e1e5d4b86a86c7d67',
 'reported_dice': 390,
 'scores': [91, 135],
 'seed': 42}
El presupuesto efectivo original seguía en 300 aunque se cambiara la variable global.


## 2. MODIFICA: completar reglas y corregir decisiones

Se conservan las 13 categorías y la estructura del profesor. Se corrige la evaluación de la última repetición, se pasan los parámetros explícitamente y se cuentan únicamente los dados nuevos. Se añaden bono superior, Yahtzees adicionales y Joker. Consulte docs/MODIFICACIONES.md para ver cada cambio.

La IA mira **una tirada futura** y vuelve a decidir tras una tirada real; no optimiza toda la partida. Cada evaluación usa un generador separado del de los dados reales.

![Flujo adaptado](resultados/flujo_adaptado.svg)

In [4]:
state = play_game(seed=SEED, sims_per_mask=SIMS_PER_MASK)
print_scoreboard(state)
print('Turnos completos:', len(state.turns))
assert len(state.turns) == 26
assert all(1 <= len(t['rolls']) <= 3 for t in state.turns)
assert all(v is not None for p in state.players for v in p.scores.values())

Categoría             Jugador 1  Jugador 2
Unos                          2          0
Doses                         4          4
Treses                        3          6
Cuatros                      12          8
Cincos                       15         10
Seises                       12         12
Trío                         21         27
Póquer                       27         29
Full                         25         25
Escalera corta               30         30
Escalera larga               40          0
Yahtzee                       0          0
Libre                        23         26
Bono superior                 0          0
Bonos Yahtzee                 0          0
TOTAL                       214        177
Ganador: Jugador 1
Dados individuales nuevos: 223
Turnos completos: 26


In [5]:
print('Primer turno (dados conservados y caras nuevas):')
pprint(state.turns[0])
print('Histograma solo de caras nuevas:', dict(state.stats.face_hist))

Primer turno (dados conservados y caras nuevas):
{'category': 'threeOfKind',
 'player': 'Jugador 1',
 'points': 21,
 'rolls': [{'dice': [6, 1, 1, 6, 3],
            'kept': [False, False, False, False, False],
            'new_faces': [6, 1, 1, 6, 3]},
           {'dice': [6, 2, 2, 6, 3],
            'kept': [True, False, False, True, True],
            'new_faces': [2, 2]},
           {'dice': [6, 2, 6, 6, 1],
            'kept': [True, False, False, True, False],
            'new_faces': [2, 6, 1]}],
 'round': 1,
 'total': 21,
 'total_increment': 21}
Histograma solo de caras nuevas: {6: 42, 1: 39, 3: 35, 2: 40, 5: 39, 4: 28}


## 3. USA: convergencia de un experimento fijo

Se conservan dos treses y se lanzan tres dados. El pago es el mejor puntaje entre Treses, Trío, Full y Libre. Las categorías no se agotan en este experimento; se repite siempre el mismo estado.

Su referencia exacta se obtiene enumerando 216 resultados equiprobables. La Ley de los Grandes Números predice convergencia de la media al valor esperado bajo el modelo IID. No obliga al error a disminuir en cada punto.

Las siguientes celdas leen la ejecución entregada. Cambie REGENERAR a True para volver a simular, incluida la comparación de presupuestos.

In [6]:
REGENERAR = False
if REGENERAR:
    results = run_experiments(n=N, seed=SEED, sims_per_mask=SIMS_PER_MASK)
else:
    results = json.loads(Path('resultados/resultados.json').read_text(encoding='utf-8'))
mean, variance = exact_fixed_ev()
print('Valor exacto:', mean, '| Varianza:', variance)
for row in results['convergence']:
    print(f"N={row['n']:>6}: media={row['ev']:.6f}; error={row['absolute_error']:.6f}; SE={row['theoretical_se']:.6f}")

Valor exacto: 17.300925925925945 | Varianza: 13.117776920438473
N=    10: media=17.500000; error=0.199074; SE=1.145329
N=   100: media=17.710000; error=0.409074; SE=0.362185
N=  1000: media=17.272000; error=0.028926; SE=0.114533
N=  5000: media=17.278800; error=0.022126; SE=0.051221
N= 10000: media=17.303600; error=0.002674; SE=0.036218
N= 50000: media=17.319720; error=0.018794; SE=0.016197
N=100000: media=17.293530; error=0.007396; SE=0.011453


![Convergencia](resultados/convergencia.svg)

![Error y error estándar](resultados/error.svg)

El valor más grande de N no se usa como verdad teórica. Tampoco se confunde el número de muestras IID con el presupuesto por máscara de la IA.

In [7]:
print('Comparación de presupuestos en un mismo estado:')
pprint(results['budget'])
print('Los tiempos son medidas del equipo; la máscara elegida no demuestra optimalidad.')

Comparación de presupuestos en un mismo estado:
[{'elapsed_s': 0.0018103000184055418, 'mask': '00101', 'sims_per_mask': 10},
 {'elapsed_s': 0.13503420000779442, 'mask': '00101', 'sims_per_mask': 1000},
 {'elapsed_s': 17.12896400000318, 'mask': '00101', 'sims_per_mask': 100000}]
Los tiempos son medidas del equipo; la máscara elegida no demuestra optimalidad.


## 4. MODIFICA: dados cargados

Se fija P(6)=0,30 y P(1)=...=P(5)=0,14. Para una tirada independiente de cinco dados, P(Yahtzee)=suma de p_i elevado a 5. Esto no representa la probabilidad de un turno con hasta tres tiradas y selección de dados. Se informa intervalo de Wilson al 95 % para cada proporción.

In [8]:
print('Uniforme:', FAIR, '| Cargado:', LOADED)
print('P exacta uniforme:', theoretical_yahtzee(FAIR))
print('P exacta cargada:', theoretical_yahtzee(LOADED))
pprint(results['distributions'])

Uniforme: (0.16666666666666666, 0.16666666666666666, 0.16666666666666666, 0.16666666666666666, 0.16666666666666666, 0.16666666666666666) | Cargado: (0.14, 0.14, 0.14, 0.14, 0.14, 0.3)
P exacta uniforme: 0.0007716049382716047
P exacta cargada: 0.0026989119999999995
[{'ci95_high': 0.0009065269130703536,
  'ci95_low': 0.0005718308846305667,
  'estimate': 0.00072,
  'n_hands': 100000,
  'scenario': 'uniforme',
  'theory': 0.0007716049382716047,
  'yahtzees': 72},
 {'ci95_high': 0.0029353020100888407,
  'ci95_low': 0.002302911354308767,
  'estimate': 0.0026,
  'n_hands': 100000,
  'scenario': 'cargado',
  'theory': 0.0026989119999999995,
  'yahtzees': 260}]


![Caras](resultados/caras.svg)

## 5. CREA: conservar un trío o arriesgar dos dados

Estado explícito: [6,6,6,1,2], queda una tirada y solo Libre está vacía. Todas las demás casillas tienen cero, incluida Yahtzee; no hay bonos. Conservar da 21 puntos. Al reservar los tres seises y lanzar los otros dos, E[Libre]=18+2 E[D].

La probabilidad de completar Yahtzee es p_6²; mide el evento, aunque la casilla Yahtzee ya no esté disponible. Si Yahtzee tuviera 50 podrían existir bonos adicionales y cambiaría el pago. La decisión es neutral al riesgo y maximiza puntos esperados.

In [9]:
for name, probabilities in [('uniforme', FAIR), ('cargado', LOADED)]:
    print(name, exact_risk(probabilities))
pprint(results['risk'])

uniforme {'stop': 21, 'reroll_ev': 25.0, 'yahtzee_probability': 0.027777777777777776, 'lose_probability': 0.027777777777777776, 'tie_probability': 0.05555555555555555}
cargado {'stop': 21, 'reroll_ev': 25.8, 'yahtzee_probability': 0.09, 'lose_probability': 0.019600000000000003, 'tie_probability': 0.039200000000000006}
[{'lose_exact': 0.027777777777777776,
  'lose_mc': 0.02905,
  'n': 100000,
  'reroll_ev_exact': 25.0,
  'reroll_ev_mc': 24.98487,
  'scenario': 'uniforme',
  'stop': 21,
  'tie_exact': 0.05555555555555555,
  'tie_mc': 0.05584,
  'yahtzee_exact': 0.027777777777777776,
  'yahtzee_mc': 0.02769},
 {'lose_exact': 0.019600000000000003,
  'lose_mc': 0.01994,
  'n': 100000,
  'reroll_ev_exact': 25.8,
  'reroll_ev_mc': 25.78493,
  'scenario': 'cargado',
  'stop': 21,
  'tie_exact': 0.039200000000000006,
  'tie_mc': 0.03934,
  'yahtzee_exact': 0.09,
  'yahtzee_mc': 0.08907}]


## 6. Conclusión y defensa

En el experimento fijo, las muestras grandes se aproximan a una referencia exacta independiente. En una trayectoria el error puede aumentar: compare 10 000 y 50 000 muestras. Los dados cargados modifican el límite al que converge la simulación; simular más no elimina ese sesgo.

Montecarlo y análisis combinatorio se complementan: la enumeración verifica casos pequeños y Montecarlo permite estudiar decisiones y escenarios más amplios. No hay precisión automática del 99,9 %.

El informe técnico y las fuentes están en docs/Informe_Montecarlo.docx. Para repetir las pruebas: python -m unittest discover -s tests -v.